# 영화 리뷰 감성 분류기의 Amazon 상품 리뷰 적응

HATESLOP 9주차 Language 과제입니다. 영화 리뷰의 감성 분류를 학습한 DistilBERT에 공개 Amazon 상품 리뷰를 LoRA로 추가 학습시킨 뒤, 상품 리뷰의 긍정·부정 분류 성능을 얼마나 바꿀 수 있는지 확인합니다.

Colab에서 재현할 때는 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 첫 셀부터 실행합니다. 두 평가는 고정된 Amazon Test 500건에 같은 입력·전처리·추론 설정을 사용합니다.

## 1. 실험 설정

In [1]:
TASK = "Language"
MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
MODEL_REVISION = "714eb0fa89d2f80546fda750413ed43d93601a13"
DATASET_ID = "fancyzhx/amazon_polarity"
DATASET_REVISION = "9d9c45c18f8c3cf1b23a3c27917b60cbf28f3289"

SEED = 42
TRAIN_SIZE = 2_000
VALIDATION_SIZE = 400
TEST_SIZE = 500
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
OUTPUT_DIR = "./assignment_runs/amazon_review_lora"

입력은 영어 상품 리뷰 본문이고 정답은 해당 리뷰의 긍정·부정 레이블입니다. SST-2로 학습된 약 6,700만 파라미터의 DistilBERT를 사용하면 기존 분류 헤드로 기초 성능을 측정할 수 있습니다. Amazon 리뷰는 별점 1–2를 부정, 4–5를 긍정으로 변환한 데이터로, 공식 Train과 Test가 제공됩니다. 두 자료 모두 Hugging Face에서 공개되어 있고 양쪽 카드에 Apache 2.0으로 표시됩니다.

[DistilBERT 모델 카드](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [Amazon Polarity 데이터셋 카드](https://huggingface.co/datasets/fancyzhx/amazon_polarity)

## 2. 실행 환경

라이브러리 버전을 고정하고 PyTorch는 실행 환경의 설치본을 사용합니다. 이 노트북은 CUDA, Apple MPS, CPU 순서로 사용 가능한 장치를 선택합니다.

In [2]:
%pip install -q transformers==4.57.3 datasets==4.4.1 peft==0.18.0 accelerate==1.12.0 pandas==2.3.3

Note: you may need to restart the kernel to use updated packages.


In [3]:
import importlib.metadata
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, disable_progress_bars, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

set_seed(SEED)
disable_progress_bars()
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("실행 환경", platform.platform(), platform.machine(), platform.python_version())
print("연산 장치", torch.cuda.get_device_name(0) if device.type == "cuda" else device)
for package in ("torch", "transformers", "datasets", "peft", "accelerate", "pandas"):
    print(package, importlib.metadata.version(package))

/tmp/hateslop-week9.JBUNn7/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


실행 환경 macOS-26.5.1-arm64-arm-64bit arm64 3.11.17
연산 장치 mps
torch 2.9.1
transformers 4.57.3
datasets 4.4.1
peft 0.18.0
accelerate 1.12.0
pandas 2.3.3


## 3. 데이터 로딩과 분할

공식 Train과 Test에 고정 시드로 버퍼 셔플을 적용하고 각 클래스에서 같은 수의 리뷰를 가져옵니다. Train에서 뽑은 2,400건은 제공된 분할 함수를 바탕으로 클래스 비율을 유지하면서 Train 2,000건과 Validation 400건으로 나눕니다. 공식 Test에서 뽑은 500건은 두 번의 최종 평가에만 씁니다.

모델 입력인 본문을 공백과 대소문자 기준으로 정규화해서 빈 리뷰와 중복 본문을 제외합니다. Test를 뽑을 때 Train/Validation 후보와 겹치는 본문도 제외합니다.

In [4]:
def prepare_splits(data, seed=SEED, validation_size=0.1):
    splits = DatasetDict({"train": data}) if isinstance(data, Dataset) else DatasetDict(data)
    if "train" not in splits:
        raise ValueError("학습용 분할의 이름을 train으로 지정하세요.")
    if "test" not in splits:
        split = splits["train"].train_test_split(
            test_size=0.1, seed=seed, stratify_by_column="label"
        )
        splits["train"], splits["test"] = split["train"], split["test"]
        validation_size = 1 / 9
    if "validation" not in splits:
        split = splits["train"].train_test_split(
            test_size=validation_size, seed=seed, stratify_by_column="label"
        )
        splits["train"], splits["validation"] = split["train"], split["test"]
    return DatasetDict({name: splits[name] for name in ("train", "validation", "test")})


def review_text(example):
    return " ".join(example["content"].split())


def sample_reviews(stream, size, seen):
    per_label = size // 2
    selected = []
    counts = {0: 0, 1: 0}
    for example in stream.shuffle(seed=SEED, buffer_size=10_000):
        label = example["label"]
        text_key = review_text(example).lower()
        if not text_key or text_key in seen or counts[label] >= per_label:
            continue
        selected.append(example)
        seen.add(text_key)
        counts[label] += 1
        if len(selected) == size:
            break
    if len(selected) != size:
        raise ValueError(f"요청한 {size}건을 뽑지 못했습니다. 실제 {len(selected)}건")
    return Dataset.from_list(selected, features=stream.features)


def load_data():
    source = load_dataset(DATASET_ID, revision=DATASET_REVISION, streaming=True)
    seen = set()
    train = sample_reviews(source["train"], TRAIN_SIZE + VALIDATION_SIZE, seen)
    test = sample_reviews(source["test"], TEST_SIZE, seen)
    return DatasetDict(train=train, test=test)


raw = prepare_splits(load_data(), validation_size=VALIDATION_SIZE)
split_counts = pd.DataFrame(
    {name: pd.Series(split["label"]).value_counts() for name, split in raw.items()}
).T.rename(columns={0: "부정", 1: "긍정"})
split_counts["합계"] = split_counts.sum(axis=1)
print(raw["train"].features)
display(split_counts)
review_keys = {
    name: {review_text(row).lower() for row in split} for name, split in raw.items()
}
for name, keys in review_keys.items():
    assert len(keys) == len(raw[name])
assert all(
    review_keys[a].isdisjoint(review_keys[b])
    for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))
)
print("분할 내·분할 간 정규화 본문 중복 0건")
print("Train 예시", review_text(raw["train"][0])[:240], "| label =", raw["train"][0]["label"])

{'label': ClassLabel(names=['negative', 'positive']), 'title': Value('string'), 'content': Value('string')}


,부정,긍정,합계
train,1000,1000,2000
validation,200,200,400
test,250,250,500


분할 내·분할 간 정규화 본문 중복 0건
Train 예시 I thought this would be a great DVD for my 5 month old son. He sometimes likes a DVD to entertain him and us.The video spends way too much time on foreign languages, and not enough English. I would like my son to learn a foreign language, b | label = 0


## 4. 모델과 전처리

리뷰 본문의 연속 공백을 정리하고 DistilBERT 토크나이저로 최대 128토큰까지 변환합니다. 배치마다 길이가 가장 긴 입력에 맞춰 패딩합니다. 별점에서 변환된 데이터셋의 0·1 레이블은 정수로 전달합니다.

In [5]:
def load_model_and_processor():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, use_safetensors=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    return model, tokenizer


model, processor = load_model_and_processor()
assert model.config.id2label == {0: "NEGATIVE", 1: "POSITIVE"}


def preprocess(example):
    encoded = processor(review_text(example), truncation=True, max_length=MAX_LENGTH)
    return {**encoded, "labels": example["label"]}


prepared = raw.map(preprocess, remove_columns=raw["train"].column_names)
example = prepared["train"][0]
print("전처리 예시", {name: value[:12] if isinstance(value, list) else value
                    for name, value in example.items()})

전처리 예시 {'input_ids': [101, 1045, 2245, 2023, 2052, 2022, 1037, 2307, 4966, 2005, 2026, 1019], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0}


분류 헤드는 SST-2 체크포인트의 가중치를 불러와 그대로 사용합니다. 데이터셋 0은 모델의 NEGATIVE, 1은 POSITIVE에 대응합니다. 이 매핑을 학습 전후 모두 고정합니다.

## 5. 학습 전 평가

Test 전체를 text-classification pipeline으로 추론합니다. 같은 본문 처리, 토크나이저, 최대 길이, 배치 크기와 장치를 두 평가에서 사용합니다. 제공된 Accuracy 함수로 정답과 예측을 비교합니다.

In [6]:
def get_targets(test_data):
    return list(test_data["label"])


def predict_labels(current_model, processor, test_data):
    # DistilBERT 본체의 활성 LoRA 층을 포함한 채 pipeline으로 전달합니다.
    inference_model = (
        current_model.get_base_model() if isinstance(current_model, PeftModel) else current_model
    )
    inference_model.eval()
    classifier = pipeline(
        "text-classification", model=inference_model, tokenizer=processor, device=device
    )
    label_ids = inference_model.config.label2id
    outputs = classifier(
        [review_text(example) for example in test_data],
        batch_size=EVAL_BATCH_SIZE,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    return [label_ids[output["label"]] for output in outputs]


def accuracy(targets, predictions):
    if len(targets) == 0 or len(targets) != len(predictions):
        raise ValueError("정답과 예측은 길이가 같고 비어 있지 않아야 합니다.")
    return sum(a == b for a, b in zip(targets, predictions)) / len(targets)

def evaluate_model(current_model, processor, test_data):
    targets = get_targets(test_data)
    predictions = predict_labels(current_model, processor, test_data)
    if len(targets) != len(test_data):
        raise ValueError("Test의 모든 샘플에 대해 정답과 예측을 반환하세요.")
    metrics = {"accuracy": accuracy(targets, predictions)}
    table = pd.DataFrame({"sample_id": range(len(targets)), "target": targets,
                          "prediction": predictions})
    return metrics, table

before_metrics, before = evaluate_model(model, processor, raw["test"])
print(before_metrics)
display(before.head())

Device set to use mps


{'accuracy': 0.882}


,sample_id,target,prediction
0,0,1,1
1,1,0,0
2,2,0,0
3,3,0,0
4,4,0,0


## 6. LoRA 적용

DistilBERT의 self-attention에 있는 query·value 선형층에 LoRA를 추가합니다. 작은 rank로 추가 파라미터를 제한하고 SST-2 분류 헤드는 고정합니다.

In [7]:
def make_lora_config(model):
    targets = ["q_lin", "v_lin"]
    matched = [name for name, _ in model.named_modules() if name.split(".")[-1] in targets]
    assert len(matched) == 2 * model.config.n_layers, matched
    print("LoRA 적용 모듈", matched)
    return LoraConfig(
        target_modules=targets,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        # 범용 PeftModel로 감싸 기존 분류 헤드도 동결 상태를 유지합니다.
    )


lora_config = make_lora_config(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable_names = [name for name, param in model.named_parameters() if param.requires_grad]
assert trainable_names and all("lora_" in name for name in trainable_names)
print("동결 확인", "LoRA 가중치만 학습")

LoRA 적용 모듈 ['distilbert.transformer.layer.0.attention.q_lin', 'distilbert.transformer.layer.0.attention.v_lin', 'distilbert.transformer.layer.1.attention.q_lin', 'distilbert.transformer.layer.1.attention.v_lin', 'distilbert.transformer.layer.2.attention.q_lin', 'distilbert.transformer.layer.2.attention.v_lin', 'distilbert.transformer.layer.3.attention.q_lin', 'distilbert.transformer.layer.3.attention.v_lin', 'distilbert.transformer.layer.4.attention.q_lin', 'distilbert.transformer.layer.4.attention.v_lin', 'distilbert.transformer.layer.5.attention.q_lin', 'distilbert.transformer.layer.5.attention.v_lin']
trainable params: 147,456 || all params: 67,102,466 || trainable%: 0.2197
동결 확인 LoRA 가중치만 학습


6개 attention 블록 각각의 query와 value에 rank 8을 적용합니다. 학습 전후 차이를 분류 헤드 재학습과 섞지 않도록 LoRA 가중치만 업데이트되는지 코드로 확인합니다.

## 7. 학습

AdamW로 2 epoch 학습하고 epoch마다 Validation Accuracy와 Loss를 기록합니다. 마지막 epoch의 Adapter로 Test를 평가합니다.

In [8]:
def make_trainer(model, processor, train_data, validation_data):
    def compute_metrics(eval_prediction):
        predictions = np.argmax(eval_prediction.predictions, axis=-1)
        return {"accuracy": accuracy(eval_prediction.label_ids, predictions)}

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=2e-4,
        num_train_epochs=2,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        label_names=["labels"],
        remove_unused_columns=False,
        dataloader_pin_memory=device.type == "cuda",
        seed=SEED,
        report_to="none",
        disable_tqdm=True,
    )
    print("학습 설정", {name: getattr(args, name) for name in (
        "optim", "learning_rate", "num_train_epochs", "per_device_train_batch_size",
        "per_device_eval_batch_size", "eval_strategy", "seed",
    )})
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=validation_data,
        processing_class=processor,
        data_collator=DataCollatorWithPadding(processor),
        compute_metrics=compute_metrics,
    )


trainer = make_trainer(model, processor, prepared["train"], prepared["validation"])
trainer.train()

학습 설정 {'optim': <OptimizerNames.ADAMW_TORCH_FUSED: 'adamw_torch_fused'>, 'learning_rate': 0.0002, 'num_train_epochs': 2, 'per_device_train_batch_size': 16, 'per_device_eval_batch_size': 16, 'eval_strategy': <IntervalStrategy.EPOCH: 'epoch'>, 'seed': 42}


{'loss': 0.4224, 'grad_norm': 1.6732345819473267, 'learning_rate': 0.00010080000000000001, 'epoch': 1.0}


{'eval_loss': 0.31649765372276306, 'eval_accuracy': 0.8775, 'eval_runtime': 1.0793, 'eval_samples_per_second': 370.61, 'eval_steps_per_second': 23.163, 'epoch': 1.0}


{'loss': 0.2951, 'grad_norm': 1.150101661682129, 'learning_rate': 8.000000000000001e-07, 'epoch': 2.0}


{'eval_loss': 0.2909061312675476, 'eval_accuracy': 0.8825, 'eval_runtime': 0.9464, 'eval_samples_per_second': 422.633, 'eval_steps_per_second': 26.415, 'epoch': 2.0}
{'train_runtime': 24.9028, 'train_samples_per_second': 160.625, 'train_steps_per_second': 10.039, 'train_loss': 0.3587186279296875, 'epoch': 2.0}


TrainOutput(global_step=250, training_loss=0.3587186279296875, metrics={'train_runtime': 24.9028, 'train_samples_per_second': 160.625, 'train_steps_per_second': 10.039, 'train_loss': 0.3587186279296875, 'epoch': 2.0})

## 8. 학습 후 평가

같은 Test 500건으로 재평가합니다. 비교표의 sample_id는 고정된 Test 순서에서의 행 번호입니다. 처음 8건을 함께 보여주고, 전체 예측을 개선·악화·계속 오답·계속 정답으로 나눠 집계합니다.

In [9]:
after_metrics, after = evaluate_model(model, processor, raw["test"])
assert len(before) >= 5 and len(after) >= 5, "비교 샘플을 최소 5개 기록하세요."
assert before["sample_id"].is_unique and after["sample_id"].is_unique
assert before[["sample_id", "target"]].reset_index(drop=True).equals(
    after[["sample_id", "target"]].reset_index(drop=True)
), "학습 전후 동일한 샘플·순서·정답을 사용하세요."
comparison = before.rename(columns={"prediction": "before"}).merge(
    after[["sample_id", "prediction"]].rename(columns={"prediction": "after"}),
    on="sample_id", validate="one_to_one"
)

score_table = pd.DataFrame({"before": before_metrics, "after": after_metrics})
score_table["change_pp"] = (score_table["after"] - score_table["before"]) * 100
display(score_table.round(4))

label_names = {0: "부정", 1: "긍정"}
before_correct = comparison["before"].eq(comparison["target"])
after_correct = comparison["after"].eq(comparison["target"])
comparison["outcome"] = np.select(
    [~before_correct & after_correct, before_correct & ~after_correct, ~after_correct],
    ["개선", "악화", "계속 오답"],
    default="계속 정답",
)
case_table = comparison.copy()
case_table["review_excerpt"] = [review_text(row)[:180] for row in raw["test"]]
for column in ("target", "before", "after"):
    case_table[column] = case_table[column].map(label_names)
with pd.option_context("display.max_colwidth", None):
    display(case_table.head(8))

outcome_order = ["개선", "악화", "계속 오답", "계속 정답"]
display(pd.crosstab(case_table["target"], case_table["outcome"])
        .reindex(columns=outcome_order, fill_value=0))
examples = case_table[case_table["outcome"] != "계속 정답"].drop_duplicates("outcome")
with pd.option_context("display.max_colwidth", None):
    display(examples)

Device set to use mps


,before,after,change_pp
accuracy,0.882,0.876,-0.6


,sample_id,target,before,after,outcome,review_excerpt
0,0,긍정,긍정,긍정,계속 정답,"If you like super-natural mysteries, then you'd like this film. It is technically a horror film, but it's really not that frightening (though there is a scene in the movie that mad"
1,1,부정,부정,부정,계속 정답,"HiI bought this shaver based on customer reviews,,So much for trusting anonymous reviews.. First of all it's cheaply made.....Motor is loud and slow compared to good shavers...had"
2,2,부정,부정,부정,계속 정답,It's amazing to me that there could be an MP3 player that allows one to corrupt it by simply uploading slightly more data that it shows. That's what happened. Pathetic. Playng volu
3,3,부정,부정,긍정,악화,My children love the content but I can't let them read the book for too long.Double the size of the font and print it in a larger format would be better.
4,4,부정,부정,부정,계속 정답,"i purchased a sony clie pegt615 on 08-15-02 at compusa/madisonWi. I found that it didn't beam and took it back for an exchange model, same item only to find that this one didn't be"
5,5,긍정,부정,부정,계속 오답,"I owned another OXO peeler for about 10 years, which is why I purchased this one. I can't believe how much better it is! It works very smoothly & rapidly. Has a great grip, even if"
6,6,긍정,긍정,긍정,계속 정답,"As a loyal fan of the Stargate franchise since the very first movie...I applaud the actors, writers and producers/director for a job well done!It still takes some getting used to a"
7,7,부정,부정,부정,계속 정답,This grill will work only as a baker and not as a hot grill that will sear steaks. It preheats after 30 minutes to about 375 degrees and the temperature drops significantly when me


outcome,개선,악화,계속 오답,계속 정답
target,,,,
긍정,8,0,28,214
부정,0,11,23,216


,sample_id,target,before,after,outcome,review_excerpt
3,3,부정,부정,긍정,악화,My children love the content but I can't let them read the book for too long.Double the size of the font and print it in a larger format would be better.
5,5,긍정,부정,부정,계속 오답,"I owned another OXO peeler for about 10 years, which is why I purchased this one. I can't believe how much better it is! It works very smoothly & rapidly. Has a great grip, even if"
33,33,긍정,부정,긍정,개선,CD in perfect condition. I was hoping the other songs were as good as the theme song. Would perchase only if really into the background songs used on the show.


## 9. 저장과 분석

Adapter, 토크나이저, 모델 설정을 저장한 뒤 같은 Base model에서 Adapter를 다시 읽어 다섯 건의 예측을 확인합니다.

In [10]:
adapter_dir = Path(OUTPUT_DIR) / "adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
model.config.save_pretrained(adapter_dir)
print("Base model", MODEL_ID, MODEL_REVISION)
print("Adapter·토크나이저·모델 설정 저장 위치", adapter_dir.resolve())
saved_files = sorted(path.name for path in adapter_dir.iterdir() if path.is_file())
print("저장 파일", saved_files)
assert {"adapter_config.json", "adapter_model.safetensors", "config.json", "tokenizer.json"}.issubset(saved_files)

restored_tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
restored_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, use_safetensors=True
)
restored_model = PeftModel.from_pretrained(restored_base, adapter_dir)
check_set = raw["test"].select(range(5))
assert predict_labels(restored_model, restored_tokenizer, check_set) == after["prediction"].head(5).tolist()
print("Adapter 재로딩 후 Test 첫 5건의 예측 일치")

Device set to use mps


Base model distilbert/distilbert-base-uncased-finetuned-sst-2-english 714eb0fa89d2f80546fda750413ed43d93601a13
Adapter·토크나이저·모델 설정 저장 위치 /private/tmp/hateslop-week9.JBUNn7/run/assignment_runs/amazon_review_lora/adapter
저장 파일 ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']


Adapter 재로딩 후 Test 첫 5건의 예측 일치


### 결과 분석

macOS Apple MPS에서 Train 2,000건으로 2 epoch를 학습했습니다. Validation Accuracy는 1 epoch에서 87.75%, 2 epoch에서 88.25%였습니다. DistilBERT의 분류 헤드를 고정했고 전체 67,102,466개 파라미터 중 query·value의 LoRA 가중치 147,456개(0.22%)만 학습했습니다.

| 동일한 Test 500건 | LoRA 적용 전 | LoRA 학습 후 | 변화 |
| --- | ---: | ---: | ---: |
| 전체 Accuracy | 88.2% (441/500) | 87.6% (438/500) | −0.6%p |
| 긍정 정답률 | 85.6% (214/250) | 88.8% (222/250) | +3.2%p |
| 부정 정답률 | 90.8% (227/250) | 86.4% (216/250) | −4.4%p |

- **개선 · ID 33** — CD 상태를 칭찬하면서 다른 수록곡에 실망한 리뷰입니다. 부정에서 긍정으로 바뀌어 정답과 일치했습니다.
- **악화 · ID 3** — 책의 내용을 좋아하지만 글자가 작아 오래 읽기 어렵다는 리뷰입니다. 부정에서 긍정으로 바뀌면서 정답을 놓쳤습니다.
- **계속 오답 · ID 5** — 새 필러의 작동과 그립을 칭찬하는 리뷰를 두 모델 모두 부정으로 분류했습니다.

정답으로 바뀐 8건은 모두 긍정 리뷰였고, 오답으로 바뀐 11건은 모두 부정 리뷰였습니다. 학습 후 긍정으로 분류한 리뷰가 19건 늘었습니다. ID 3과 33처럼 칭찬과 불만이 함께 나온 사례에서 긍정 쪽으로 바뀌는 모습이 있었고, 전체 Accuracy는 3건 차이로 하락했습니다.

평가는 고정 시드로 뽑은 클래스 균형 표본에서 한 번 수행했습니다. 중립 별점은 원래 데이터셋에서 빠져 있고 128토큰 이후는 잘리므로, 다른 표본이나 전체 쇼핑 리뷰에서도 같은 경향이 나타날지는 이 실험으로 확인하기 어렵습니다. 중복 검사는 정규화한 본문이 일치하는 경우를 대상으로 했습니다.

Base model은 위에 표시한 DistilBERT 체크포인트입니다. 실행 시 현재 작업 디렉터리의 `assignment_runs/amazon_review_lora/adapter`에 Adapter, 토크나이저, 모델 설정이 저장됩니다. 저장한 Adapter를 불러온 뒤 Test 첫 5건의 예측이 학습 직후 결과와 일치했습니다.